# Reduced MultiTask GP

高次元入力をPCAで圧縮しながら task identity を保持する MultiTask / Kronecker の考え方を確認します。


## このNotebookで学ぶこと

- **PCAによる入力次元削減**と**MultiTask GPによるタスク間の情報共有**を区別する
- long-formatデータの作り方と `task_feature` の意味を理解する
- `model.fit()` による学習後のPosteriorを可視化する
- 事後平均による候補選択と、**qLogEIによるタスク固定の候補最適化**を比較する
- 候補を元の5次元の目的関数で評価する

**注意：** モデル内部のPCA潜在空間は2次元ですが、候補探索の設計変数は元の5次元です。タスクIDは連続的に最適化する設計変数ではありません。

## Long-format MultiTask


### long-formatとPCAの役割

各観測を `[x0, x1, x2, x3, x4, task_id]` の6列で表します。タスク0は `x0 + x1`、タスク1は `x0 - x1` を目的関数とします。

PCAは5つの連続特徴量を2次元へ縮約しますが、タスクIDを連続特徴量として縮約しません。異なるタスクの観測点 `X0` と `X1` は一致している必要がありません。

In [ ]:
import torch
from robotorchan.models import PCAMultiTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X0 = torch.rand(8, 5)
X1 = torch.rand(8, 5)
train_X = torch.cat([
    torch.cat([X0, torch.zeros(8, 1)], dim=-1),
    torch.cat([X1, torch.ones(8, 1)], dim=-1),
])
train_Y = torch.cat([X0[:, :1] + X0[:, 1:2], X1[:, :1] - X1[:, 1:2]])
model = PCAMultiTaskGP(train_X, train_Y, task_feature=-1, n_components=2)
print(model.raw_train_X.shape)


## Posterior


### 学習前後のPosterior

次のPosterior呼び出しは、まだGPのハイパーパラメータを最適化していない状態の確認です。実際の予測・候補提案には、後続の `model.fit()` による学習後のモデルを使います。

In [ ]:
X_test = torch.cat([torch.rand(3, 5), torch.zeros(3, 1)], dim=-1)
print(model.posterior(X_test).mean.squeeze(-1))


## Fit the multi-task GP

タスク列を保持したまま、既存のExact MLLで学習します。


In [ ]:

model.fit()
model.eval()
print("fitted task-aware model")


### 学習済みPosteriorをタスク別に可視化

5次元のうち `x0` だけを変え、残りの4変数を0.5に固定した断面を描きます。タスク0とタスク1の真の関数、事後平均、潜在関数の近似95%事後区間を比較します。

この区間は測定ノイズを含む観測予測区間ではありません。PCAで失われた情報の影響もあり、断面の予測精度は保証されません。

In [ ]:
import matplotlib.pyplot as plt

slice_features = torch.full((121, 5), 0.5)
slice_features[:, 0] = torch.linspace(0.0, 1.0, len(slice_features))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for task_id, ax in enumerate(axes):
    task_column = torch.full((len(slice_features), 1), float(task_id))
    slice_input = torch.cat([slice_features, task_column], dim=-1)
    with torch.no_grad():
        post = model.posterior(slice_input)
        mean = post.mean.squeeze(-1)
        std = post.variance.clamp_min(0).sqrt().squeeze(-1)
    true_slice = (slice_features[:, 0] + slice_features[:, 1]
                  if task_id == 0 else slice_features[:, 0] - slice_features[:, 1])
    ax.plot(slice_features[:, 0], true_slice, "--", label="true slice")
    ax.plot(slice_features[:, 0], mean, label="posterior mean")
    ax.fill_between(slice_features[:, 0], mean - 1.96 * std,
                    mean + 1.96 * std, alpha=0.2,
                    label="approx. 95% latent interval")
    ax.set_title(f"task {task_id}")
    ax.set_xlabel("x0 (x1..x4 = 0.5)")
axes[0].set_ylabel("objective")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### タスク別の残差と予測不確実性

上の図では平均と95%潜在区間を確認しました。次に、同じ断面で**予測平均 − 真値**と**潜在関数の事後標準偏差**を描きます。残差は既知の合成関数とのずれであり、事後標準偏差はモデルが見積もる不確実性です。両者が一致するとは限りません。

タスク0と1では (x_1) の寄与が反対です。PCA縮約によってその違いがどの程度再現されるかを、過大な性能主張をせず確認します。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for task_id in (0, 1):
    query = torch.cat(
        [slice_features, torch.full((len(slice_features), 1), float(task_id))],
        dim=-1,
    )
    with torch.no_grad():
        task_post = model.posterior(query)
        predicted = task_post.mean.squeeze(-1)
        uncertainty = task_post.variance.clamp_min(0).sqrt().squeeze(-1)
    true_values = (
        slice_features[:, 0] + slice_features[:, 1]
        if task_id == 0
        else slice_features[:, 0] - slice_features[:, 1]
    )
    residual = predicted - true_values
    axes[0].plot(slice_features[:, 0], residual, label=f"Task {task_id}")
    axes[1].plot(slice_features[:, 0], uncertainty, label=f"Task {task_id}")
    print(f"Task {task_id} slice RMSE: {residual.square().mean().sqrt().item():.4f}")
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_ylabel("Posterior mean - truth")
axes[1].set_ylabel("Posterior standard deviation")
axes[1].set_xlabel("Original x0 (x1..x4 = 0.5)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## Task-specific candidate proposals

各タスクを固定した候補プールを元の5次元空間で評価し、posterior mean最大の候補を提案します。タスクIDは最適化対象ではありません。


In [ ]:
candidate_features = torch.rand(128, 5)
for task_id in (0, 1):
    task_column = torch.full((128, 1), float(task_id))
    pool = torch.cat([candidate_features, task_column], dim=-1)
    with torch.no_grad():
        scores = model.posterior(pool).mean.squeeze(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    features = candidate[:, :5]
    if task_id == 0:
        observed = features[:, :1] + features[:, 1:2]
    else:
        observed = features[:, :1] - features[:, 1:2]
    assert candidate.shape == torch.Size([1, 6])
    assert candidate[0, -1].item() == task_id
    print("task:", task_id, "candidate:", candidate)
    print("posterior mean:", scores.max(), "objective:", observed)


### 事後平均による候補選択

既存の実装では、各タスクのIDを固定した128点の候補プールについて、Posterior Meanが最大となる点を選びます。これは活用（exploitation）を中心とした基準であり、獲得関数の連続最適化ではありません。

以下では、この比較例を残したうえで、BoTorchのqLogEIを使った探索を追加します。

### qLogEIによるタスク別候補最適化

`qLogExpectedImprovement` は予測の不確実性も利用して改善の期待値を評価します。`optimize_acqf()` の `fixed_features={5: task_id}` によってタスクID列を固定し、残りの5次元を最適化します。

**`best_f` は対象タスクで得られた最大観測値**を使います。異なるタスクの目的値を混ぜて改善量を評価しないためです。ここでは各タスクの合成目的関数で提案候補を評価します。

このMultiTaskモデルは複数タスクを持つため、BoTorchのMC獲得関数には明示的なスカラー目的関数 `GenericMCObjective` を渡します。タスクIDを固定した入力に対するPosteriorの単一出力を選択し、`model.num_outputs` が複数でもqLogEIを構築できるようにします。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import GenericMCObjective
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

task_proposals = {}
search_bounds = torch.stack([torch.zeros(6), torch.ones(6)])
for task_id in (0, 1):
    task_observations = train_Y[train_X[:, 5] == task_id]
    task_acqf = qLogExpectedImprovement(
        model=model,
        objective=GenericMCObjective(lambda samples, X=None: samples[..., 0]),
        best_f=task_observations.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    proposed, acq_value = optimize_acqf(
        acq_function=task_acqf,
        bounds=search_bounds,
        q=1,
        num_restarts=3,
        raw_samples=32,
        fixed_features={5: float(task_id)},
    )
    task_proposals[task_id] = proposed.detach().clone()
    assert proposed.shape == (1, 6)
    assert proposed[0, 5].item() == task_id
    features = proposed[:, :5]
    observed = (features[:, :1] + features[:, 1:2]
                if task_id == 0 else features[:, :1] - features[:, 1:2])
    print(f"task {task_id} qLogEI candidate:", proposed)
    print("acquisition:", acq_value, "evaluated objective:", observed)

### タスクごとの qLogEI 提案位置

既存の `optimize_acqf` は各タスクIDを固定して最適化します。下図では**提案された (x_0,x_1)** の位置と学習観測点を示します。残りの次元は図に表示しないため、2次元の散布図だけで候補の優劣を判断できません。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for task_id in (0, 1):
    task_mask = train_X[:, 5] == task_id
    ax.scatter(
        train_X[task_mask, 0],
        train_X[task_mask, 1],
        alpha=0.6,
        label=f"Task {task_id} observations",
    )
    ax.scatter(
        task_proposals[task_id][0, 0],
        task_proposals[task_id][0, 1],
        marker="*",
        s=180,
        label=f"Task {task_id} qLogEI proposal",
    )
ax.set_xlabel("Original x0")
ax.set_ylabel("Original x1")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## まとめ

| 処理 | 使用するAPI・考え方 |
|---|---|
| モデル構築 | `PCAMultiTaskGP(..., task_feature=-1, n_components=2)` |
| GP学習 | `model.fit()` |
| タスク別予測 | タスクID列を含む入力で `posterior(X)` |
| 比較用候補選択 | 有限プールのPosterior Mean最大化 |
| BOの候補最適化 | `qLogExpectedImprovement` + `optimize_acqf(fixed_features=...)` |
| 候補評価 | タスク別の合成目的関数 |

タスクIDを固定して候補探索を行うことで、PCAで縮約したGPを使いつつ、元の5次元設計空間でベイズ最適化の候補を提案できます。今回は1回の候補提案までで、観測追加・再学習の反復ループは含みません。